# 08 · Does the fire signal survive if we stop averaging it away?

Notebook 05 ran the proper before/after test on the Marshall Fire and got a clean negative:
whole-chip embedding shift showed essentially no correlation with actual burn severity
(dNBR), either immediately after the fire (r=-0.06) or by 2024 (r=-0.01). The leading
explanation offered there was dilution -- a chip here is 2.24km across, a fire can burn one
corner of it and leave the rest untouched, and mean-pooling every patch token into one vector
before comparing embeddings blends the burned part away, the same way one burnt strawberry
doesn't make a whole blended smoothie taste burnt.

This notebook tests that explanation directly rather than leaving it as a plausible-sounding
guess. Clay's encoder already computes a token per spatial patch before `encode_batch()`
averages them together -- `src/clay_embed.py` now also exposes `encode_batch_patches()`,
which returns those tokens unpooled. The test:

1. Refetch the exact same three dates and AOI as notebook 05 (pre-fire, immediate post-fire,
   2024 long-term) and recompute the same chip-level dNBR ground truth.
2. Embed all three dates at **patch resolution** instead of chip resolution.
3. Block-average the full-resolution dNBR raster down to the same patch grid, so every patch
   token has its own matching burn-severity value.
4. Correlate patch-level embedding shift against patch-level dNBR, pooled across every patch
   in every chip -- a much larger, spatially precise version of notebook 05's chip-level test.
5. As a secondary, geometry-light check: does the *single most-shifted patch* in a chip track
   that chip's overall dNBR better than the chip's averaged embedding did?

If dilution was the real explanation, patch-level correlation should come out clearly
stronger than notebook 05's chip-level r ~ -0.06. If it doesn't, that's an equally real
result -- it would mean Clay's embeddings just don't carry a burn signal here at any
resolution, and dilution was the wrong story.

A T4 GPU runtime makes this fast, but isn't required -- this AOI is small enough
(~50 chips x 3 dates) to also run on a plain CPU runtime if GPU quota is exhausted, just
slower. Self-contained, same small AOI as notebook 05 -- no Drive mount, doesn't touch
`docs/data/chips.geojson`.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import torch
from scipy import stats

from src import clay_embed, stac_utils

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print("No GPU detected -- continuing on CPU. This AOI is small (~50 chips x 3 dates),")
    print("so a CPU-only run should still finish, just noticeably slower than a T4 would be.")

os.makedirs("docs/figures", exist_ok=True)
catalog = stac_utils.open_catalog()
BBOX = stac_utils.MARSHALL_FIRE_BBOX
print(f"AOI: {BBOX}, device: {device}")

## Refetch the same three dates as notebook 05

Same AOI, same date windows, same `select_clearest_scene()` snow/cloud-aware selection --
this is a fresh Colab VM, so nothing from notebook 05's run persists, but the selection is
deterministic given the same catalog query and should land on the same scenes. The true-color
QA check below still matters: confirm it before trusting anything downstream.

In [ ]:
pre_item = stac_utils.select_clearest_scene(catalog, BBOX, "2021-11-01/2021-12-29", label="pre-fire")
post_item = stac_utils.select_clearest_scene(catalog, BBOX, "2022-01-01/2022-03-31", label="immediate post-fire")
longterm_item = stac_utils.select_clearest_scene(catalog, BBOX, "2024-06-01/2024-09-15", label="long-term (2024)")

In [ ]:
def load_scene(item):
    ds = odc.stac.load(
        [item], bands=stac_utils.S2_BANDS, bbox=BBOX,
        crs="EPSG:32613", resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
    )
    arr_da = ds.to_array(dim="band")
    arr = arr_da.isel(time=0).compute() if "time" in arr_da.dims else arr_da.compute()
    return arr, ds

pre_arr, pre_ds = load_scene(pre_item)
post_arr, post_ds = load_scene(post_item)
longterm_arr, longterm_ds = load_scene(longterm_item)

print("shapes:", pre_arr.shape, post_arr.shape, longterm_arr.shape)
assert pre_arr.shape == post_arr.shape == longterm_arr.shape, "grids didn't align -- check bbox/crs/resolution"

In [ ]:
def true_color(arr):
    rgb = arr.sel(band=["B04", "B03", "B02"]).values.astype("float32")
    out = np.zeros_like(rgb)
    for i in range(3):
        lo, hi = np.nanpercentile(rgb[i], [2, 98])
        out[i] = np.clip((rgb[i] - lo) / (hi - lo + 1e-6), 0, 1)
    return out.transpose(1, 2, 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, arr, item, label in zip(
    axes, [pre_arr, post_arr, longterm_arr], [pre_item, post_item, longterm_item],
    ["pre-fire", "immediate post-fire", "long-term (2024)"],
):
    ax.imshow(true_color(arr))
    ax.set_title(f"{label}\n{item.datetime.date()}")
    ax.axis("off")
plt.tight_layout()
plt.savefig("docs/figures/fire_patch_true_color_dates.png", dpi=150, bbox_inches="tight")
plt.show()
print("Check all three panels for snow/washout, and confirm these dates match notebook 05's")
print("run, before trusting anything quantitative below.")

## Chip grid and dNBR (same as notebook 05)

Same NBR/dNBR definition, same chip grid, same 5% nodata threshold. This time each chip's
pixel slice is also kept in `chips_meta` -- notebook 05 didn't need it, but the patch-level
test below needs the exact pixel window to block-average dNBR down to the patch grid.

In [ ]:
NIR, SWIR2 = 6, 9  # indices into stac_utils.S2_BANDS

def nbr(arr):
    nir = arr.values[NIR].astype("float32")
    swir2 = arr.values[SWIR2].astype("float32")
    return (nir - swir2) / (nir + swir2 + 1e-6)

nbr_pre = nbr(pre_arr)
nbr_post = nbr(post_arr)
dnbr = nbr_pre - nbr_post

height, width = pre_arr.shape[1], pre_arr.shape[2]
grid = stac_utils.make_pixel_chip_grid(height, width)
print(f"{len(grid)} candidate chips ({height // stac_utils.CHIP_SIZE_PX} rows x {width // stac_utils.CHIP_SIZE_PX} cols)")

x_coords, y_coords = pre_arr.x.values, pre_arr.y.values
raster_crs = pre_ds.odc.crs
NODATA_FRAC_THRESHOLD = 0.05

chips_meta, pre_pixels, post_pixels, longterm_pixels, mean_dnbr = [], [], [], [], []
for chip in grid:
    ys, xs = chip["y_slice"], chip["x_slice"]
    pre_patch = pre_arr.values[:, ys, xs]
    post_patch = post_arr.values[:, ys, xs]
    lt_patch = longterm_arr.values[:, ys, xs]
    if max(np.isnan(pre_patch).mean(), np.isnan(post_patch).mean(), np.isnan(lt_patch).mean()) > NODATA_FRAC_THRESHOLD:
        continue

    bounds = stac_utils.pixel_window_to_lonlat_bounds(x_coords, y_coords, chip, raster_crs)
    lat, lon = stac_utils.bounds_centroid(bounds)

    chips_meta.append({"id": chip["id"], "bounds": bounds, "lat": lat, "lon": lon, "y_slice": ys, "x_slice": xs})
    pre_pixels.append(np.nan_to_num(pre_patch, nan=0.0).astype("float32"))
    post_pixels.append(np.nan_to_num(post_patch, nan=0.0).astype("float32"))
    longterm_pixels.append(np.nan_to_num(lt_patch, nan=0.0).astype("float32"))
    mean_dnbr.append(float(np.nanmean(dnbr[ys, xs])))

pre_pixels = np.stack(pre_pixels)
post_pixels = np.stack(post_pixels)
longterm_pixels = np.stack(longterm_pixels)
mean_dnbr = np.array(mean_dnbr)
print(f"Kept {len(chips_meta)}/{len(grid)} chips present in all three dates")

## Embed at patch resolution

`encode_batch_patches()` (added to `src/clay_embed.py` for this notebook) returns Clay's
per-patch tokens before the mean-pool that `encode_batch()` normally applies. Running this
once per date gives us both things we need: the unpooled tokens for the patch-level test, and
-- by averaging them ourselves with `.mean(axis=1)` -- the exact same whole-chip embedding
`encode_batch()` would have produced, so the chip-level baseline recomputed below is numerically
identical to notebook 05's approach, not just the same idea.

In [ ]:
ckpt_path = clay_embed.download_checkpoint()
metadata_path = clay_embed.download_metadata_yaml()
model = clay_embed.load_model(ckpt_path, metadata_path, device=device)
wavelengths, band_means, band_stds = clay_embed.load_band_stats(metadata_path)
print("Clay v1.5 loaded")

In [ ]:
def embed_all_patches(pixels, scene_date, chips_meta, batch_size=16):
    dates = [pd.Timestamp(scene_date)] * len(chips_meta)
    lats = [m["lat"] for m in chips_meta]
    lons = [m["lon"] for m in chips_meta]
    out = []
    for start in range(0, len(chips_meta), batch_size):
        end = min(start + batch_size, len(chips_meta))
        batch_pixels = clay_embed.normalize_chips(pixels[start:end], band_means, band_stds)
        time_feats, latlon_feats = clay_embed.make_time_latlon_tensors(
            dates[start:end], lats[start:end], lons[start:end]
        )
        out.append(clay_embed.encode_batch_patches(model, batch_pixels, time_feats, latlon_feats, wavelengths, device=device))
    return np.concatenate(out, axis=0)  # (n_chips, n_patches, embed_dim)

patches_pre = embed_all_patches(pre_pixels, pre_item.datetime, chips_meta)
patches_post = embed_all_patches(post_pixels, post_item.datetime, chips_meta)
patches_longterm = embed_all_patches(longterm_pixels, longterm_item.datetime, chips_meta)

n_chips, n_patches, embed_dim = patches_pre.shape
patch_side = round(n_patches ** 0.5)
assert patch_side * patch_side == n_patches, (
    f"n_patches={n_patches} isn't a perfect square -- Clay's patch grid may not be square, "
    "or the class token wasn't fully stripped. Inspect encode_batch_patches()'s output shape "
    "directly before trusting the block-averaging below."
)
assert stac_utils.CHIP_SIZE_PX % patch_side == 0, (
    f"chip size {stac_utils.CHIP_SIZE_PX} doesn't divide evenly by patch_side={patch_side} -- "
    "the block-average reshape below assumes it does."
)
print(f"Embedded {n_chips} chips x 3 dates, {n_patches} patches/chip ({patch_side}x{patch_side}), dim={embed_dim}")

## Baseline: reproduce notebook 05's chip-level result, from this run's own data

Averaging the same patch tokens the patch-level test below uses -- so this number and the
patch-level number are directly comparable, not just similar experiments run separately.

In [ ]:
def cosine_sim_rows(a, b):
    a_n = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_n = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (a_n * b_n).sum(axis=1)

chip_pre = patches_pre.mean(axis=1)
chip_post = patches_post.mean(axis=1)
chip_longterm = patches_longterm.mean(axis=1)

shift_immediate_chip = 1 - cosine_sim_rows(chip_pre, chip_post)
shift_longterm_chip = 1 - cosine_sim_rows(chip_pre, chip_longterm)

r_chip_immediate = stats.pearsonr(mean_dnbr, shift_immediate_chip)
r_chip_longterm = stats.pearsonr(mean_dnbr, shift_longterm_chip)

print(f"[chip-level baseline] dNBR vs. shift (pre -> immediate): r={r_chip_immediate[0]:.3f}, p={r_chip_immediate[1]:.4f}")
print(f"[chip-level baseline] dNBR vs. shift (pre -> 2024):      r={r_chip_longterm[0]:.3f}, p={r_chip_longterm[1]:.4f}")
print("Compare against notebook 05's r=-0.06 (immediate) / r=-0.01 (2024) as a sanity check")
print("that this rerun landed on equivalent scenes/dNBR.")

## The real test: patch-level dNBR vs. patch-level embedding shift

For each chip, block-average its full-resolution dNBR raster down to the same
`patch_side x patch_side` grid as the patch tokens (an exact reshape+mean since
`CHIP_SIZE_PX` divides evenly by `patch_side`), then flatten every chip's patches into one
long array. This trades chip count (~50) for patch count (tens of thousands of patches once
run), a much larger and spatially precise sample than notebook 05's chip-level test could offer.

In [ ]:
block = stac_utils.CHIP_SIZE_PX // patch_side

def patch_dnbr_grid(chip_meta):
    sub = dnbr[chip_meta["y_slice"], chip_meta["x_slice"]]  # (CHIP_SIZE_PX, CHIP_SIZE_PX)
    return np.nanmean(sub.reshape(patch_side, block, patch_side, block), axis=(1, 3)).reshape(-1)

patch_dnbr_all = np.concatenate([patch_dnbr_grid(m) for m in chips_meta])  # (n_chips * n_patches,)

def patch_shift_all(a, b):
    a_n = a / np.linalg.norm(a, axis=2, keepdims=True)
    b_n = b / np.linalg.norm(b, axis=2, keepdims=True)
    return (1 - (a_n * b_n).sum(axis=2)).reshape(-1)  # (n_chips * n_patches,)

patch_shift_immediate_all = patch_shift_all(patches_pre, patches_post)
patch_shift_longterm_all = patch_shift_all(patches_pre, patches_longterm)

valid = ~np.isnan(patch_dnbr_all)
r_patch_immediate = stats.pearsonr(patch_dnbr_all[valid], patch_shift_immediate_all[valid])
r_patch_longterm = stats.pearsonr(patch_dnbr_all[valid], patch_shift_longterm_all[valid])

print(f"n = {valid.sum()} patches across {n_chips} chips")
print(f"[patch-level] dNBR vs. shift (pre -> immediate): r={r_patch_immediate[0]:.3f}, p={r_patch_immediate[1]:.4g}")
print(f"[patch-level] dNBR vs. shift (pre -> 2024):      r={r_patch_longterm[0]:.3f}, p={r_patch_longterm[1]:.4g}")
print()
print("If |r_patch| is meaningfully larger than |r_chip| above, that supports the dilution")
print("explanation for notebook 05's null result. If it's similarly weak, dilution wasn't it --")
print("Clay's embeddings likely just don't carry a burn signal here at any resolution tested.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
rng = np.random.default_rng(0)
sample = rng.choice(valid.sum(), size=min(8000, valid.sum()), replace=False)

axes[0].scatter(patch_dnbr_all[valid][sample], patch_shift_immediate_all[valid][sample], s=4, alpha=0.25, color="#f58231")
axes[0].set_xlabel("patch dNBR (burn severity)")
axes[0].set_ylabel("patch embedding shift, pre -> immediate")
axes[0].set_title(f"Immediate (r={r_patch_immediate[0]:.2f}, n={valid.sum()})")

axes[1].scatter(patch_dnbr_all[valid][sample], patch_shift_longterm_all[valid][sample], s=4, alpha=0.25, color="#5ec8ff")
axes[1].set_xlabel("patch dNBR (burn severity)")
axes[1].set_ylabel("patch embedding shift, pre -> 2024")
axes[1].set_title(f"Long-term (r={r_patch_longterm[0]:.2f}, n={valid.sum()})")

plt.tight_layout()
plt.savefig("docs/figures/fire_patch_dnbr_vs_shift.png", dpi=150)
plt.show()
print("(plotting a random 8k-point subsample for readability -- r/p above are computed on all patches)")

## Secondary, geometry-light check: does the single most-shifted patch explain the chip?

The block-average test above assumes patch tokens line up with the input grid in simple
row-major order -- true for a standard ViT encoder, but worth a check that doesn't depend on
getting that geometry exactly right. Simpler question: within each chip, does the *single most
different* patch (max shift) track that chip's overall dNBR better than the chip's averaged
embedding did? If dilution is real, the max should surface the burned sub-region even in chips
where the average washes it out.

In [ ]:
max_patch_shift_immediate = patch_shift_immediate_all.reshape(n_chips, n_patches).max(axis=1)
max_patch_shift_longterm = patch_shift_longterm_all.reshape(n_chips, n_patches).max(axis=1)

r_max_immediate = stats.pearsonr(mean_dnbr, max_patch_shift_immediate)
r_max_longterm = stats.pearsonr(mean_dnbr, max_patch_shift_longterm)

print(f"[max-patch, per chip] dNBR vs. shift (pre -> immediate): r={r_max_immediate[0]:.3f}, p={r_max_immediate[1]:.4f}")
print(f"[max-patch, per chip] dNBR vs. shift (pre -> 2024):      r={r_max_longterm[0]:.3f}, p={r_max_longterm[1]:.4f}")

In [ ]:
labels = ["chip mean\n(notebook 05 style)", "patch-level\n(all patches)", "chip max-patch"]
immediate_rs = [r_chip_immediate[0], r_patch_immediate[0], r_max_immediate[0]]
longterm_rs = [r_chip_longterm[0], r_patch_longterm[0], r_max_longterm[0]]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.bar(x - width/2, immediate_rs, width, label="pre -> immediate post-fire", color="#f58231")
ax.bar(x + width/2, longterm_rs, width, label="pre -> 2024 long-term", color="#5ec8ff")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Pearson r (dNBR vs. embedding shift)")
ax.set_title("Does zooming into patches recover the fire signal?")
ax.legend()
plt.tight_layout()
plt.savefig("docs/figures/fire_patch_vs_chip_comparison.png", dpi=150)
plt.show()

## One chip, visually: does its patch-level shift map line up with its dNBR map?

Picks the chip with the highest mean dNBR (the most-burned chip in the AOI) and lays its
dNBR sub-grid next to its patch-level immediate-shift grid, both at the same NxN patch-grid
resolution, so any spatial correspondence (or lack of one) is visible directly rather than
only as a correlation coefficient.

In [ ]:
worst_idx = int(np.argmax(mean_dnbr))
worst_meta = chips_meta[worst_idx]
worst_dnbr_grid = patch_dnbr_grid(worst_meta).reshape(patch_side, patch_side)
worst_shift_grid = patch_shift_immediate_all.reshape(n_chips, n_patches)[worst_idx].reshape(patch_side, patch_side)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(true_color(pre_arr.isel(
    y=worst_meta["y_slice"], x=worst_meta["x_slice"],
)))
axes[0].set_title(f"Chip {worst_meta['id']} (pre-fire, true color)")
axes[0].axis("off")

im1 = axes[1].imshow(worst_dnbr_grid, cmap="RdYlGn_r")
axes[1].set_title("dNBR, block-averaged to patch grid")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], shrink=0.7)

im2 = axes[2].imshow(worst_shift_grid, cmap="inferno")
axes[2].set_title("Patch embedding shift, pre -> immediate")
axes[2].axis("off")
fig.colorbar(im2, ax=axes[2], shrink=0.7)

plt.tight_layout()
plt.savefig("docs/figures/fire_patch_worst_chip_detail.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDone. Commit the new docs/figures/fire_patch_*.png files back to the repo.")
print("(This notebook doesn't touch docs/data/chips.geojson -- self-contained AOI, same as notebook 05.)")

## Follow-up: what's actually behind the negative max-patch correlation?

The secondary check above found something genuinely strange: chips that burned *less*
severely tended to have a *larger* single-patch embedding shift, not a smaller one (r =
&minus;0.55) &mdash; backwards from a burn signal. One concrete, testable guess, raised but not
tested above: this is snow or cloud-shadow contamination showing up in an unrelated patch, not
anything about the fire.

Sentinel-2's Scene Classification Layer (SCL) gives a per-pixel land/cloud/snow classification
computed independently of Clay's embedding and independently of dNBR &mdash; exactly the kind
of ground truth needed to test this without just eyeballing thumbnails.
`stac_utils.SCL_BAD_CLASSES` (cloud shadow, cloud, cirrus, snow/ice) already exists and is used
elsewhere in this project for date selection. This fetches SCL for the same pre-fire and
immediate-post-fire scenes already loaded above, and checks whether each chip's single
most-shifted patch has an unusually high fraction of SCL-flagged pixels that *changed
classification* between the two dates &mdash; the specific pattern that would produce a large,
fire-irrelevant embedding shift.

In [ ]:
from src.stac_utils import SCL_BAD_CLASSES

def load_scl(item):
    ds = odc.stac.load(
        [item], bands=["SCL"], bbox=BBOX,
        crs="EPSG:32613", resolution=stac_utils.GSD_M, resampling="nearest",
    )
    scl_da = ds["SCL"]
    arr = scl_da.isel(time=0).values if "time" in scl_da.dims else scl_da.values
    return arr.astype("int16")

scl_pre = load_scl(pre_item)
scl_post = load_scl(post_item)
print("SCL loaded for pre-fire and immediate post-fire scenes, shape:", scl_pre.shape)

In [ ]:
bad_classes = list(SCL_BAD_CLASSES)

def patch_changed_frac_grid(chip_meta):
    ys, xs = chip_meta["y_slice"], chip_meta["x_slice"]
    bad_pre = np.isin(scl_pre[ys, xs], bad_classes)
    bad_post = np.isin(scl_post[ys, xs], bad_classes)
    changed = bad_pre != bad_post  # SCL classification flipped between the two dates
    return changed.astype("float32").reshape(patch_side, block, patch_side, block).mean(axis=(1, 3)).reshape(-1)

max_patch_idx_immediate = patch_shift_immediate_all.reshape(n_chips, n_patches).argmax(axis=1)

max_patch_changed_frac = np.array([
    patch_changed_frac_grid(chips_meta[c])[max_patch_idx_immediate[c]]
    for c in range(n_chips)
])
typical_changed_frac = np.array([
    patch_changed_frac_grid(chips_meta[c]).mean()
    for c in range(n_chips)
])

r_dnbr_changed = stats.pearsonr(mean_dnbr, max_patch_changed_frac)
r_shift_changed = stats.pearsonr(max_patch_shift_immediate, max_patch_changed_frac)

print(f"Max-shifted patch's SCL-changed pixel fraction (bad-class flipped between dates):")
print(f"  mean at the max-shift patch: {max_patch_changed_frac.mean():.3f}")
print(f"  mean at a typical patch in the same chip: {typical_changed_frac.mean():.3f}")
print(f"  correlation with chip dNBR:              r={r_dnbr_changed[0]:.3f}, p={r_dnbr_changed[1]:.4f}")
print(f"  correlation with the max-shift itself:    r={r_shift_changed[0]:.3f}, p={r_shift_changed[1]:.4f}")
print()
print("If the max-shift patch's changed-classification fraction is well above a typical patch's,")
print("and correlates negatively with dNBR (low-burn chips having noisier classification) and")
print("positively with the shift itself, that supports snow/cloud-shadow contamination as the")
print("real driver of the unexplained max-patch result -- not anything about the fire.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(mean_dnbr, max_patch_changed_frac, s=18, alpha=0.7, color="#3cb44b")
axes[0].set_xlabel("chip mean dNBR (burn severity)")
axes[0].set_ylabel("max-shift patch's SCL-changed fraction")
axes[0].set_title(f"vs. dNBR (r={r_dnbr_changed[0]:.2f})")

axes[1].scatter(max_patch_shift_immediate, max_patch_changed_frac, s=18, alpha=0.7, color="#911eb4")
axes[1].set_xlabel("max-patch embedding shift")
axes[1].set_ylabel("max-shift patch's SCL-changed fraction")
axes[1].set_title(f"vs. max-patch shift (r={r_shift_changed[0]:.2f})")

plt.tight_layout()
plt.savefig("docs/figures/fire_patch_snow_diagnostic.png", dpi=150)
plt.show()

## One example, visually: the chip with the noisiest max-shift patch

If the SCL-changed-fraction story holds, the chip with the highest max-shift patch and the
highest changed-fraction should visibly show snow or cloud-shadow appearing or disappearing at
that specific location between the pre-fire and immediate-post-fire dates.

In [ ]:
noisiest_idx = int(np.argmax(max_patch_changed_frac))
noisiest_meta = chips_meta[noisiest_idx]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(true_color(pre_arr.isel(y=noisiest_meta["y_slice"], x=noisiest_meta["x_slice"])))
axes[0].set_title(f"Chip {noisiest_meta['id']} (pre-fire)\ndNBR={mean_dnbr[noisiest_idx]:.3f}, max-patch shift={max_patch_shift_immediate[noisiest_idx]:.3f}")
axes[0].axis("off")

axes[1].imshow(true_color(post_arr.isel(y=noisiest_meta["y_slice"], x=noisiest_meta["x_slice"])))
axes[1].set_title("Same chip (immediate post-fire)")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("docs/figures/fire_patch_snow_diagnostic_example_chip.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDone. Commit docs/figures/fire_patch_snow_diagnostic*.png back to the repo.")